## 10 — Plot Generation (Milestone 2)

This notebook loads the model comparison table produced by `09_evaluation_comparison.ipynb`
and generates all required comparison plots.

### Inputs
- `../results/all_model_comparison.csv` — from `09_evaluation_comparison.ipynb`
- `../results/transformer_results.pkl` — predictions from Transformer model
- `../results/bilstm_results.pkl` — predictions from BiLSTM model

### Outputs
- `../plots/comparison_binary_f1.png`  — binary F1-score comparison bar chart
- `../plots/comparison_macro_f1.png`   — macro-averaged F1-score comparison bar chart
- `../plots/nlp_best_confusion_matrix.png` — confusion matrix for best NLP model

## 1. Imports

Initializes the environment, sets the matplotlib backend to 'Agg' for non-interactive saving, and sets up the directory structure.

In [11]:
import pickle
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from pathlib import Path

RESULTS_DIR = Path("../results")
PLOTS_DIR   = Path("../plots")
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print("Imports done.")

Imports done.


## 2. Load Comparison Table
Loads the all_model_comparison.csv file which contains the performance metrics for both baseline and novel models.

In [12]:
all_results = pd.read_csv(RESULTS_DIR / "all_model_comparison.csv")

print(f"Models loaded: {len(all_results)}")
print(all_results[["model", "type", "binary_f1_pct", "macro_f1_pct"]].to_string(index=False))

Models loaded: 8
                model             type  binary_f1_pct  macro_f1_pct
FFT-MLP (All Sensors) Paper (Baseline)          90.93         46.12
    FFT-MLP (IMU+THM) Paper (Baseline)          90.86         41.07
     CNN-BiLSTM (TOF) Paper (Baseline)          91.60         29.50
          Late Fusion Paper (Baseline)          94.28         50.14
  Intermediate Fusion Paper (Baseline)          91.11         39.61
    FFT-Random Forest Paper (Baseline)          89.81         38.80
    Transformer (NLP)      NLP (Novel)          90.93         85.98
         BiLSTM (NLP)      NLP (Novel)          89.98         84.19


## 3. Color Scheme

Defines a consistent color palette to distinguish between the paper baselines and the novel NLP models.

Blue bars = Paper (Baseline) models  
Salmon bars = NLP (Novel) models

In [13]:
colors = [
    "steelblue" if t == "Paper (Baseline)" else "salmon"
    for t in all_results["type"]
]

legend_elements = [
    Patch(facecolor="steelblue", label="Paper Models (Baseline)"),
    Patch(facecolor="salmon",    label="NLP Models (Novel)")
]

print("Color scheme defined.")

Color scheme defined.


## 4. Plot 1 — Binary F1-Score Comparison

Generates a bar chart comparing the Binary F1-Score across all models.

In [14]:
fig, ax = plt.subplots(figsize=(12, 6))

bars = ax.bar(
    all_results["model"],
    all_results["binary_f1_pct"],
    color=colors,
    edgecolor="white",
    width=0.6
)

for bar, val in zip(bars, all_results["binary_f1_pct"]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f"{val:.2f}%",
        ha="center", va="bottom", fontsize=10, fontweight="bold"
    )

ax.legend(handles=legend_elements, fontsize=12)
ax.set_xlabel("Model", fontsize=13)
ax.set_ylabel("Binary F1-Score (%)", fontsize=13)
ax.set_title("Binary F1-Score Comparison: Paper Models vs NLP Models", fontsize=14)
ax.set_ylim(0, 115)
ax.set_xticklabels(all_results["model"], rotation=30, ha="right", fontsize=11)
ax.yaxis.set_tick_params(labelsize=11)

plt.tight_layout()
plt.savefig(PLOTS_DIR / "comparison_binary_f1.png", dpi=150)
plt.close()
print("Saved: ../plots/comparison_binary_f1.png")

Saved: ../plots/comparison_binary_f1.png


/var/folders/0h/3zx9xjy179g6p34hk4l_0wqc0000gn/T/ipykernel_5731/3914681608.py:24: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(all_results["model"], rotation=30, ha="right", fontsize=11)


## 5. Plot 2 — Macro-Averaged F1-Score Comparison

Generates a bar chart comparing the Macro-Averaged F1-Score, highlighting the significant performance jump in the novel NLP models.

In [15]:
fig, ax = plt.subplots(figsize=(12, 6))

bars = ax.bar(
    all_results["model"],
    all_results["macro_f1_pct"],
    color=colors,
    edgecolor="white",
    width=0.6
)

for bar, val in zip(bars, all_results["macro_f1_pct"]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f"{val:.2f}%",
        ha="center", va="bottom", fontsize=10, fontweight="bold"
    )

ax.legend(handles=legend_elements, fontsize=12)
ax.set_xlabel("Model", fontsize=13)
ax.set_ylabel("Macro-Averaged F1-Score (%)", fontsize=13)
ax.set_title("Macro-Averaged F1-Score Comparison: Paper Models vs NLP Models", fontsize=14)
ax.set_ylim(0, 80)
ax.set_xticklabels(all_results["model"], rotation=30, ha="right", fontsize=11)
ax.yaxis.set_tick_params(labelsize=11)

plt.tight_layout()
plt.savefig(PLOTS_DIR / "comparison_macro_f1.png", dpi=150)
plt.close()
print("Saved: ../plots/comparison_macro_f1.png")

/var/folders/0h/3zx9xjy179g6p34hk4l_0wqc0000gn/T/ipykernel_5731/2780752488.py:24: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(all_results["model"], rotation=30, ha="right", fontsize=11)


Saved: ../plots/comparison_macro_f1.png


## 6. Plot 3 — Confusion Matrix for Best NLP Model

Identifies the best-performing NLP model (Transformer vs. BiLSTM) and generates a confusion matrix for its binary classification results.

In [16]:
# Load both NLP model results
with open(RESULTS_DIR / "transformer_results.pkl", "rb") as f:
    transformer_res = pickle.load(f)

with open(RESULTS_DIR / "bilstm_results.pkl", "rb") as f:
    bilstm_res = pickle.load(f)

# Determine best NLP model by binary F1
if transformer_res["binary_f1"] >= bilstm_res["binary_f1"]:
    best_res  = transformer_res
    best_name = "Transformer"
else:
    best_res  = bilstm_res
    best_name = "BiLSTM"

print(f"Best NLP model by Binary F1: {best_name}")
print(f"  Transformer Binary F1 : {transformer_res['binary_f1']*100:.2f}%")
print(f"  BiLSTM     Binary F1  : {bilstm_res['binary_f1']*100:.2f}%")

Best NLP model by Binary F1: Transformer
  Transformer Binary F1 : 90.93%
  BiLSTM     Binary F1  : 89.98%


In [17]:
# Build confusion matrix
labels     = best_res["labels"]   
preds      = best_res["preds"]    
class_names = ["Non-BFRB", "BFRB"]

cm = confusion_matrix(labels, preds, labels=[0, 1])

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(
    ax=ax,
    cmap="Blues",
    colorbar=False,
    values_format="d"
)

ax.set_title(
    f"Confusion Matrix — {best_name} (NLP Model)\nBinary F1: {best_res['binary_f1']*100:.2f}%",
    fontsize=13
)
ax.set_xlabel("Predicted Label", fontsize=12)
ax.set_ylabel("True Label", fontsize=12)
ax.xaxis.set_tick_params(labelsize=11)
ax.yaxis.set_tick_params(labelsize=11)

plt.tight_layout()
plt.savefig(PLOTS_DIR / "nlp_best_confusion_matrix.png", dpi=150)
plt.close()
print(f"Saved: ../plots/nlp_best_confusion_matrix.png")

Saved: ../plots/nlp_best_confusion_matrix.png


## 7. 

A final confirmation step that prints the paths of all successfully generated and saved plots

In [18]:
print("=== Plot Generation Complete ===")
print("Saved: ../plots/comparison_binary_f1.png")
print("Saved: ../plots/comparison_macro_f1.png")
print(f"Saved: ../plots/nlp_best_confusion_matrix.png  ({best_name})")

=== Plot Generation Complete ===
Saved: ../plots/comparison_binary_f1.png
Saved: ../plots/comparison_macro_f1.png
Saved: ../plots/nlp_best_confusion_matrix.png  (Transformer)
